In [1]:
import os
import shutil
import pandas as pd
import numpy as np
from PIL import Image, ImageOps
from sklearn.model_selection import train_test_split
from tqdm import tqdm


In [2]:

# ========================================================
# 1. DEFINE PATHS (Adjust these if your folder names differ)
# ========================================================
METADATA_PATH = "dataset/HAM10000/HAM10000_metadata.csv"
PART1_DIR = "dataset/HAM10000/HAM10000_images_part_1"
PART2_DIR = "dataset/HAM10000/HAM10000_images_part_2"

OUTPUT_DIR = "Final_Dataset"
TRAIN_DIR = os.path.join(OUTPUT_DIR, "Train")
VAL_DIR = os.path.join(OUTPUT_DIR, "Validation")

CLASSES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
TARGET_SIZE = (224, 224)

# Create clean output directory structure
for folder in [TRAIN_DIR, VAL_DIR]:
    for cls in CLASSES:
        os.makedirs(os.path.join(folder, cls), exist_ok=True)

print("✓ Output folders structured perfectly.")



✓ Output folders structured perfectly.


In [3]:

# ========================================================
# 2. READ METADATA & STRATIFIED SPLIT
# ========================================================
df = pd.read_csv(METADATA_PATH)

# Stratified split ensures equal proportions of classes in train & validation sets
train_df, val_df = train_test_split(
    df, 
    test_size=0.20, 
    stratify=df['dx'], 
    random_state=42
)

print(f"✓ Stratified Split Complete. Training samples: {len(train_df)}, Validation samples: {len(val_df)}")

# Helper function to find where an image lives (part 1 or part 2)
def find_image_path(image_id):
    p1 = os.path.join(PART1_DIR, f"{image_id}.jpg")
    p2 = os.path.join(PART2_DIR, f"{image_id}.jpg")
    if os.path.exists(p1): return p1
    if os.path.exists(p2): return p2
    return None

✓ Stratified Split Complete. Training samples: 8012, Validation samples: 2003


In [4]:
# ========================================================
# 3. PROCESS VALIDATION SET (NO AUGMENTATION, JUST RESIZE)
# ========================================================
print("\nProcessing Validation Set (Resizing to 224x224)...")
for _, row in tqdm(val_df.iterrows(), total=len(val_df)):
    img_path = find_image_path(row['image_id'])
    if img_path:
        with Image.open(img_path) as img:
            img_resized = img.resize(TARGET_SIZE, Image.Resampling.LANCZOS)
            dest = os.path.join(VAL_DIR, row['dx'], f"{row['image_id']}.jpg")
            img_resized.save(dest)



Processing Validation Set (Resizing to 224x224)...


100%|██████████| 2003/2003 [01:10<00:00, 28.33it/s]


In [5]:


# ========================================================
# 4. PROCESS & AUGMENT TRAINING SET
# ========================================================
print("\nProcessing Training Set & Saving Original Files...")
# First, pass all original training images over to their respective folders
for _, row in tqdm(train_df.iterrows(), total=len(train_df)):
    img_path = find_image_path(row['image_id'])
    if img_path:
        with Image.open(img_path) as img:
            img_resized = img.resize(TARGET_SIZE, Image.Resampling.LANCZOS)
            dest = os.path.join(TRAIN_DIR, row['dx'], f"{row['image_id']}.jpg")
            img_resized.save(dest)

# Find out how many images the majority class ('nv') has in the training set
TARGET_COUNT = len(os.listdir(os.path.join(TRAIN_DIR, "nv")))
print(f"\nTarget count set by majority class ('nv'): {TARGET_COUNT} images per folder.")

print("\nStarting Offline Data Augmentation for Minority Classes...")
for cls in CLASSES:
    cls_folder = os.path.join(TRAIN_DIR, cls)
    current_images = os.listdir(cls_folder)
    current_count = len(current_images)
    
    if current_count >= TARGET_COUNT:
        print(f"-> Class [{cls}] already balanced ({current_count} images). Skipping augmentation.")
        continue
        
    needed_count = TARGET_COUNT - current_count
    print(f"-> Augmenting Class [{cls}]: Current count = {current_count}. Generating {needed_count} variations...")
    
    # Progress bar for generating new images for this specific class
    pbar = tqdm(total=needed_count)
    aug_idx = 0
    
    while aug_idx < needed_count:
        for orig_img_name in current_images:
            if aug_idx >= needed_count:
                break
                
            orig_path = os.path.join(cls_folder, orig_img_name)
            with Image.open(orig_path) as img:
                # Apply random operations to generate clinical variants
                # 1. Random rotations (90, 180, 270)
                rot_deg = np.random.choice([90, 180, 270])
                mod_img = img.rotate(rot_deg)
                
                # 2. Random horizontal or vertical flips
                if np.random.rand() > 0.5:
                    mod_img = ImageOps.mirror(mod_img)
                if np.random.rand() > 0.5:
                    mod_img = ImageOps.flip(mod_img)
                
                # Save as a distinct new file
                base_name = os.path.splitext(orig_img_name)[0]
                new_name = f"{base_name}_aug_{aug_idx}.jpg"
                mod_img.save(os.path.join(cls_folder, new_name))
                
                aug_idx += 1
                pbar.update(1)
    pbar.close()

print("\n========================================================")
print("SUCCESS: Your static balanced dataset is ready in 'Final_Dataset/'!")
print("========================================================")


Processing Training Set & Saving Original Files...


100%|██████████| 8012/8012 [04:40<00:00, 28.60it/s]



Target count set by majority class ('nv'): 5364 images per folder.

Starting Offline Data Augmentation for Minority Classes...
-> Augmenting Class [akiec]: Current count = 262. Generating 5102 variations...


100%|██████████| 5102/5102 [00:20<00:00, 250.69it/s]


-> Augmenting Class [bcc]: Current count = 411. Generating 4953 variations...


...

-> Augmenting Class [mel]: Current count = 890. Generating 4474 variations...


100%|██████████| 4474/4474 [00:26<00:00, 165.83it/s]


-> Class [nv] already balanced (5364 images). Skipping augmentation.
-> Augmenting Class [vasc]: Current count = 114. Generating 5250 variations...


100%|██████████| 5250/5250 [00:15<00:00, 333.58it/s]


SUCCESS: Your static balanced dataset is ready in 'Final_Dataset/'!
